In [81]:
import numpy as np
import pandas as pd

In [82]:
url="../data/Telco-Customer-Churn.csv"
df= pd.read_csv(url)
RANDOM_STATE=66

In [83]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split,GridSearchCV,learning_curve

In [84]:
def prepare_data(df):
    df = df.copy()
    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
    X = df.drop(columns=['customerID', 'Churn'])
    y = df['Churn']
    return X, y

In [85]:
X,y = prepare_data(df)

In [86]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

In [87]:
X.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges'],
      dtype='str')

In [88]:
X.head()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


In [89]:
conditions=[
    (X["tenure"]>=0)&(X["tenure"]<=6),
    (X["tenure"]>6)&(X["tenure"]<=12),
    (X["tenure"]>12)&(X["tenure"]<=24),
    (X["tenure"]>24)&(X["tenure"]<=48),
    (X["tenure"]>48)&(X["tenure"]<=60),
    (X["tenure"]>60)&(X["tenure"]<=72),
]

labels = ["0-6","7-12","13-24","25-48","49-60","61-72"]

X["TenureGroup"]=np.select(conditions,labels,default=None)

X["IsNewCustomers"]=(X["tenure"]<=6).astype(int)
X["IsLongTermCustomers"]=(X["tenure"]>=48).astype(int)

X["TotalCharges"] = pd.to_numeric(
    X["TotalCharges"],
    errors="coerce"
)

X["AvgChargePerMonth"] = (
    X["TotalCharges"] /
    X["tenure"].replace(0, np.nan)
)
X["MonthlyToAvgChargeRatio"] = (
    X["MonthlyCharges"] /
    X["AvgChargePerMonth"].replace(0, np.nan)
)

X["IsMonthToMonth"] = (
    X["Contract"] == "Month-to-month"
).astype(int)

X["HasLongContract"] = (
    X["Contract"].isin(["One year", "Two year"])
).astype(int)

service_columns = [
    "PhoneService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

X["NumServices"] = sum(
    (X[col] == "Yes").astype(int)
    for col in service_columns
)

X["HasInternet"] = (
    X["InternetService"] != "No"
).astype(int)

X["HasFiber"] = (
    X["InternetService"] == "Fiber optic"
).astype(int)

X["HasSupportServices"] = (
    (X["OnlineSecurity"] == "Yes") |
    (X["TechSupport"] == "Yes")
).astype(int)

X["HasStreaming"] = (
    (X["StreamingTV"] == "Yes") |
    (X["StreamingMovies"] == "Yes")
).astype(int)


X["AutoPayment"] = (
    X["PaymentMethod"].isin([
        "Bank transfer (automatic)",
        "Credit card (automatic)"
    ])
).astype(int)

X.drop(['tenure'],inplace=True,axis=1)

X.columns


/tmp/ipykernel_43995/1020975351.py:80: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  X.drop(['tenure'],inplace=True,axis=1)


Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges',
       'TotalCharges', 'TenureGroup', 'IsNewCustomers', 'IsLongTermCustomers',
       'AvgChargePerMonth', 'MonthlyToAvgChargeRatio', 'IsMonthToMonth',
       'HasLongContract', 'NumServices', 'HasInternet', 'HasFiber',
       'HasSupportServices', 'HasStreaming', 'AutoPayment'],
      dtype='str')

In [90]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['str']).columns.tolist() 

print(num_cols)
print(cat_cols)

['SeniorCitizen', 'MonthlyCharges', 'TotalCharges', 'IsNewCustomers', 'IsLongTermCustomers', 'AvgChargePerMonth', 'MonthlyToAvgChargeRatio', 'IsMonthToMonth', 'HasLongContract', 'NumServices', 'HasInternet', 'HasFiber', 'HasSupportServices', 'HasStreaming', 'AutoPayment']
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TenureGroup']


In [91]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   gender                   7043 non-null   str    
 1   SeniorCitizen            7043 non-null   int64  
 2   Partner                  7043 non-null   str    
 3   Dependents               7043 non-null   str    
 4   PhoneService             7043 non-null   str    
 5   MultipleLines            7043 non-null   str    
 6   InternetService          7043 non-null   str    
 7   OnlineSecurity           7043 non-null   str    
 8   OnlineBackup             7043 non-null   str    
 9   DeviceProtection         7043 non-null   str    
 10  TechSupport              7043 non-null   str    
 11  StreamingTV              7043 non-null   str    
 12  StreamingMovies          7043 non-null   str    
 13  Contract                 7043 non-null   str    
 14  PaperlessBilling         7043 non-n

In [92]:
def build_preprocessor(X: pd.DataFrame):
    continuous_cols = [
        'MonthlyCharges', 
        'TotalCharges', 
        'AvgChargePerMonth', 
        'MonthlyToAvgChargeRatio',
        
    ]
    
    binary_int_cols = [
        'SeniorCitizen', 
        'IsNewCustomers', 
        'IsLongTermCustomers', 
        'IsMonthToMonth', 
        'HasLongContract', 
        'NumServices', 
        'HasInternet', 
        'HasFiber', 
        'HasSupportServices', 
        'HasStreaming', 
        'AutoPayment'
    ]
    
    cat_cols = [
        'gender', 'Partner', 'Dependents', 'PhoneService', 
        'MultipleLines', 'InternetService', 'OnlineSecurity', 
        'OnlineBackup', 'DeviceProtection', 'TechSupport', 
        'StreamingTV', 'StreamingMovies', 'Contract', 
        'PaperlessBilling', 'PaymentMethod', 'TenureGroup'
    ]
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    return ColumnTransformer(transformers=[
        ('num', num_pipeline, continuous_cols),
        ('binary', 'passthrough', binary_int_cols),
        ('cat', cat_pipeline, cat_cols)
    ])

In [93]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV,train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
import joblib

candidates={}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
preprocessor = build_preprocessor(X)

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95, random_state=42)), 
    ('classifier', RandomForestClassifier(random_state=42,class_weight='balanced'))
])

rf_hyperparams = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2]
}

rf_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_hyperparams,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

print("Running Grid Search for Random Forest with PCA...")
rf_search.fit(X_train, y_train)
print("--- Random Forest Best Results ---")
print("Best Parameters:", rf_search.best_params_)
print("Best CV ROC-AUC:", rf_search.best_score_)

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95, random_state=42)),  # Retains 95% of total variance
    ('classifier', XGBClassifier(random_state=42, eval_metric='logloss'))
])

xgb_hyperparams = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [3, 5],
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__subsample': [0.8, 1.0]
}

xgb_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_hyperparams,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

print("\nRunning Grid Search for XGBoost with PCA...")
xgb_search.fit(X_train, y_train)
print("\n--- XGBoost Best Results ---")
print("Best Parameters:", xgb_search.best_params_)
print("Best CV ROC-AUC:", xgb_search.best_score_)

if xgb_search.best_score_ > rf_search.best_score_:
    production_model = xgb_search.best_estimator_
    candidates["XGBoost + PCA"]=production_model
    print("\nSelected Model for Production: XGBoost + PCA")
else:
    production_model = rf_search.best_estimator_
    candidates["Random Forest + PCA"]=production_model
    print("\nSelected Model for Production: Random Forest + PCA")


y_pred = production_model.predict(X_test)
y_prob = production_model.predict_proba(X_test)[:, 1]

print("\n--- Test Set Performance ---")
print(f"Test ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


model_filename = "../models/churn_pipeline_v4.joblib"
joblib.dump(production_model, model_filename)
print(f"\nProduction model successfully frozen and saved to '{model_filename}'!")

Running Grid Search for Random Forest with PCA...
--- Random Forest Best Results ---
Best Parameters: {'classifier__max_depth': 10, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}
Best CV ROC-AUC: 0.8378393418793847

Running Grid Search for XGBoost with PCA...

--- XGBoost Best Results ---
Best Parameters: {'classifier__learning_rate': 0.05, 'classifier__max_depth': 3, 'classifier__n_estimators': 200, 'classifier__subsample': 1.0}
Best CV ROC-AUC: 0.8445977681066816

Selected Model for Production: XGBoost + PCA

--- Test Set Performance ---
Test ROC-AUC: 0.8373

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.51      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409


Production model success

In [94]:
day3_champion = joblib.load('../models/churn_v3.joblib')
candidates['Day-3 Champion (RF v3)'] = day3_champion

from sklearn.metrics import classification_report, confusion_matrix, recall_score, precision_score, f1_score, roc_auc_score


rows = []
for name, model in candidates.items():
    if name == 'Day-3 Champion (RF v3)':
        X,y = prepare_data(df)
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        print(X_tr.shape)
        y_pred = model.predict(X_te)
        y_prob = model.predict_proba(X_te)[:, 1]
    else:
        print(X_train.shape)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    rows.append({
        'Model': name,
        'Recall (Churn)': recall_score(y_test, y_pred),
        'Precision (Churn)': precision_score(y_test, y_pred),
        'F1 (Churn)': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

results = pd.DataFrame(rows).round(4).sort_values('Recall (Churn)', ascending=False)
results

(5634, 31)


(5634, 19)


,Model,Recall (Churn),Precision (Churn),F1 (Churn),ROC-AUC
1,Day-3 Champion (RF v3),0.8048,0.5181,0.6304,0.8533
0,XGBoost + PCA,0.5053,0.6342,0.5625,0.8373
